# SpeedrEye Colab Training

Train YOLO on the KITTI config in 10-epoch stages up to 100 epochs. Use a GPU runtime in Colab.

## 1. Setup

Colab is not the local Conda `ai` environment, so this cell installs the runtime packages needed in Colab.

In [ ]:
!pip install -q ultralytics tensorboard pyyaml

from pathlib import Path
import torch
from ultralytics import YOLO

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


## 2. Dataset location

Keep KITTI close to the Colab runtime. Use local Colab storage for fastest temporary training, or Google Drive if the dataset/checkpoints must persist between sessions.

In [ ]:
# Optional: mount Drive if kitti.yaml or the dataset is stored there.
# from google.colab import drive
# drive.mount('/content/drive')

DATA_CONFIG = "kitti.yaml"  # Change this to your Drive path if needed.
MODEL_WEIGHTS = "yolo26n.pt"
PROJECT_DIR = Path("runs") / "speedereye"
IMAGE_SIZE = 640
STAGE_SIZE = 10
FINAL_EPOCH = 100


## 3. Train in stages

Ultralytics saves `last.pt` and `best.pt` after each stage under `runs/speedereye/.../weights/`.

In [ ]:
current_weights = MODEL_WEIGHTS

for stage_end_epoch in range(STAGE_SIZE, FINAL_EPOCH + STAGE_SIZE, STAGE_SIZE):
    model = YOLO(str(current_weights))
    stage_name = f"kitti_yolo26_stage_{stage_end_epoch:03d}"

    results = model.train(
        data=DATA_CONFIG,
        epochs=STAGE_SIZE,
        imgsz=IMAGE_SIZE,
        project=str(PROJECT_DIR),
        name=stage_name,
        exist_ok=True,
    )

    current_weights = Path(results.save_dir) / "weights" / "last.pt"
    print(f"Stage {stage_end_epoch} complete. Saved weights: {current_weights}")


## 4. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/speedereye
